In [30]:
# model
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=6,
                 max_len=256, d_ff=1024, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        self.lm_head = nn.Linear(d_model, vocab_size)

    def generate_square_subsequent_mask(self, size, device):
        mask = torch.triu(torch.ones(size, size, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float("-inf"))
        return mask

    def forward(self, x):
        """
        x: (B, T)
        returns: (B, T, vocab_size)
        """
        B, T = x.size()
        mask = self.generate_square_subsequent_mask(T, x.device)
        x = self.token_embedding(x)
        x = self.pos_encoding(x)
        x = self.transformer(x, mask=mask)
        logits = self.lm_head(x)  # (B, T, vocab_size)
        return logits

def init_weights(m):
    if isinstance(m, nn.Linear):
        torch.nn.init.normal_(m.weight, std=0.02)
        if m.bias is not None:
            torch.nn.init.zeros_(m.bias)

In [31]:
# overfit
import torch
import torch.nn as nn
import numpy as np

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 32
SEQ_LEN = 256
EPOCHS = 3000

train_data = np.load("train.npy")
small_data = train_data[:BATCH_SIZE * 2]

train_tensor = torch.tensor(small_data, dtype=torch.long).to(DEVICE)
val_tensor = train_tensor.clone()

vocab_size = int(train_tensor.max()) + 1
print(f"Vocab size: {vocab_size}")

model = DecoderOnlyTransformer(vocab_size, d_model=256, nhead=8, num_layers=6,
                               dim_feedforward=1024, dropout=0.1).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)
criterion = nn.CrossEntropyLoss()

def run_epoch(data, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0

    for i in range(0, len(data), BATCH_SIZE):
        batch = data[i:i+BATCH_SIZE]
        x = batch[:, :-1]
        y = batch[:, 1:]

        logits = model(x)
        loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))

        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item()

    return total_loss / (len(data) / BATCH_SIZE)

print("Starting overfitting check...")
for epoch in range(EPOCHS):
    train_loss = run_epoch(train_tensor, train=True)
    if epoch % 200 == 0 or epoch == EPOCHS-1:
        val_loss = run_epoch(val_tensor, train=False)
        print(f"Epoch {epoch:4d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | PPL: {torch.exp(torch.tensor(train_loss)):.2f}")

'# overfit\nimport torch\nimport torch.nn as nn\nimport numpy as np\n\nDEVICE = "cuda" if torch.cuda.is_available() else "cpu"\nBATCH_SIZE = 32\nSEQ_LEN = 256\nEPOCHS = 3000\n\n# Загружаем маленький кусок данных\ntrain_data = np.load("train.npy")\nsmall_data = train_data[:BATCH_SIZE * 2]  # небольшой датасет для оверфита\n\ntrain_tensor = torch.tensor(small_data, dtype=torch.long).to(DEVICE)\nval_tensor = train_tensor.clone()\n\nvocab_size = int(train_tensor.max()) + 1\nprint(f"Vocab size: {vocab_size}")\n\nmodel = DecoderOnlyTransformer(vocab_size, d_model=256, nhead=8, num_layers=6, \n                               dim_feedforward=1024, dropout=0.1).to(DEVICE)\n\noptimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, betas=(0.9, 0.95), weight_decay=0.1)\ncriterion = nn.CrossEntropyLoss()\n\ndef run_epoch(data, train=True):\n    model.train() if train else model.eval()\n    total_loss = 0.0\n    \n    for i in range(0, len(data), BATCH_SIZE):\n        batch = data[i:i+BATCH_SIZE]\

In [32]:
# train
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pickle
import math
import time
import os
from datetime import timedelta

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 64
GRAD_ACCUM_STEPS = 4
SEQ_LENGTH = 256
MAX_ITERS = 100000
WARMUP_STEPS = 2000

EVAL_EVERY = 1000
LOG_EVERY = 200
EARLY_STOP_PATIENCE = 5


train_data = np.load("train.npy")
val_data = np.load("val.npy")

train_data = np.concatenate(train_data)
val_data = np.concatenate(val_data)

with open("char_to_idx.pkl", "rb") as f:
    char_to_idx = pickle.load(f)
with open("idx_to_char.pkl", "rb") as f:
    idx_to_char = pickle.load(f)

vocab_size = len(char_to_idx)
print(f"Vocab size: {vocab_size} | Train tokens: {len(train_data):,} | Val tokens: {len(val_data):,}")
print(f"Training on device: {DEVICE}\n")

model = DecoderOnlyTransformer(
    vocab_size=vocab_size,
    d_model=256,
    nhead=8,
    num_layers=6,
    max_len=256,
    dropout=0.1
).to(DEVICE)

model.lm_head.weight = model.token_embedding.weight

optimizer = optim.AdamW(
    model.parameters(),
    lr=3e-4,
    betas=(0.9, 0.95),
    weight_decay=0.1
)

def get_lr(it):
    if it < WARMUP_STEPS:
        return 3e-4 * it / WARMUP_STEPS
    else:
        decay_ratio = (it - WARMUP_STEPS) / (MAX_ITERS - WARMUP_STEPS)
        coeff = 0.5 * (1 + math.cos(math.pi * decay_ratio))
        return 3e-5 + (3e-4 - 3e-5) * coeff

model.apply(init_weights)

best_val_loss = float('inf')
patience_counter = 0
iter_num = 0
start_time = time.time()

print("=== Training started (warmup + cosine decay + label smoothing 0.1) ===\n")

while iter_num < MAX_ITERS:
    model.train()
    optimizer.zero_grad()

    batch_losses = []
    accum_start = time.time()

    for _ in range(GRAD_ACCUM_STEPS):
        idx = torch.randint(0, len(train_data) - SEQ_LENGTH, (BATCH_SIZE,))
        x = torch.stack([torch.from_numpy(train_data[i:i+SEQ_LENGTH]) for i in idx]).to(DEVICE)
        y = torch.stack([torch.from_numpy(train_data[i+1:i+SEQ_LENGTH+1]) for i in idx]).to(DEVICE)

        logits = model(x)
        loss = F.cross_entropy(
            logits.reshape(-1, vocab_size),
            y.reshape(-1),
            label_smoothing=0.1
        )
        loss = loss / GRAD_ACCUM_STEPS
        loss.backward()
        batch_losses.append(loss.item() * GRAD_ACCUM_STEPS)

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()

    lr = get_lr(iter_num)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    iter_num += 1

    if iter_num % LOG_EVERY == 0 or iter_num == 1:
        avg_train_loss = sum(batch_losses) / GRAD_ACCUM_STEPS
        elapsed = time.time() - start_time
        tokens_per_sec = (BATCH_SIZE * SEQ_LENGTH * GRAD_ACCUM_STEPS * LOG_EVERY) / (elapsed + 1e-8) if iter_num > 1 else 0
        print(f"Iter {iter_num:6d} | lr {lr:.2e} | train loss {avg_train_loss:.4f} | "
              f"tokens/s {tokens_per_sec:7.0f} | elapsed {str(timedelta(seconds=int(elapsed)))}")

    if iter_num % EVAL_EVERY == 0 or iter_num == MAX_ITERS:
        model.eval()
        val_losses = []
        with torch.no_grad():
            for _ in range(20):
                idx = torch.randint(0, len(val_data) - SEQ_LENGTH, (BATCH_SIZE,))
                x = torch.stack([torch.from_numpy(val_data[i:i+SEQ_LENGTH]) for i in idx]).to(DEVICE)
                y = torch.stack([torch.from_numpy(val_data[i+1:i+SEQ_LENGTH+1]) for i in idx]).to(DEVICE)

                logits = model(x)
                loss = F.cross_entropy(logits.reshape(-1, vocab_size), y.reshape(-1), label_smoothing=0.0)
                val_losses.append(loss.item())

        val_loss = sum(val_losses) / len(val_losses)
        print(f"{'='*80}")
        print(f"Iter {iter_num:6d} | Val loss: {val_loss:.4f} | Best val: {best_val_loss:.4f}")
        print(f"{'='*80}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pth")
            print(f"   >>> New best model saved! Val loss = {val_loss:.4f}")
        else:
            patience_counter += 1
            print(f"   Patience: {patience_counter}/{EARLY_STOP_PATIENCE}")

        if patience_counter >= EARLY_STOP_PATIENCE:
            print(f"\nEarly stopping triggered after {iter_num} iterations.")
            break

torch.save(model.state_dict(), "final_model.pth")
total_time = time.time() - start_time
print(f"\nTraining completed in {str(timedelta(seconds=int(total_time)))}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Final model and best model saved.")

Vocab size: 65 | Train tokens: 1,108,736 | Val tokens: 138,496
Training on device: cuda

=== Training started (warmup + cosine decay + label smoothing 0.1) ===

Iter      1 | lr 0.00e+00 | train loss 4.1973 | tokens/s       0 | elapsed 0:00:01
Iter    200 | lr 2.98e-05 | train loss 3.4842 | tokens/s   59491 | elapsed 0:03:40
Iter    400 | lr 5.98e-05 | train loss 3.4106 | tokens/s   29837 | elapsed 0:07:19
Iter    600 | lr 8.98e-05 | train loss 3.0708 | tokens/s   19915 | elapsed 0:10:58
Iter    800 | lr 1.20e-04 | train loss 2.9272 | tokens/s   14947 | elapsed 0:14:36
Iter   1000 | lr 1.50e-04 | train loss 2.8558 | tokens/s   11964 | elapsed 0:18:15
Iter   1000 | Val loss: 2.5126 | Best val: inf
   >>> New best model saved! Val loss = 2.5126
Iter   1200 | lr 1.80e-04 | train loss 2.7249 | tokens/s    9956 | elapsed 0:21:56
Iter   1400 | lr 2.10e-04 | train loss 2.6093 | tokens/s    8535 | elapsed 0:25:35
Iter   1600 | lr 2.40e-04 | train loss 2.4895 | tokens/s    7471 | elapsed 0:29:1

In [34]:
import torch
import pickle

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_vocab():
    with open("char_to_idx.pkl", "rb") as f:
        char_to_idx = pickle.load(f)
    with open("idx_to_char.pkl", "rb") as f:
        idx_to_char = pickle.load(f)
    return char_to_idx, idx_to_char

def greedy_decode(model, start_text, char_to_idx, idx_to_char, max_length=500, temperature=1.0):
    model.eval()
    generated = [char_to_idx[ch] for ch in start_text]
    input_ids = torch.tensor([generated], dtype=torch.long, device=DEVICE)

    for _ in range(max_length):
        with torch.no_grad():
            logits = model(input_ids[:, -256:])
            next_token_logits = logits[0, -1, :] / temperature
            next_token = torch.argmax(next_token_logits).item()

        generated.append(next_token)
        input_ids = torch.cat([input_ids, torch.tensor([[next_token]], device=DEVICE)], dim=1)

    return "".join([idx_to_char[i] for i in generated])

if __name__ == "__main__":
    char_to_idx, idx_to_char = load_vocab()
    vocab_size = len(char_to_idx)

    model = DecoderOnlyTransformer(vocab_size, d_model=256, nhead=8, num_layers=6,
                                   d_ff=1024, dropout=0.1).to(DEVICE)

    model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))

    prompt = "First Citizen: Before we proceed any further, hear me speak."
    generated_text = greedy_decode(model, prompt, char_to_idx, idx_to_char, max_length=600)

    print("=== GENERATED TEXT (Greedy) ===\n")
    print(generated_text)

=== GENERATED TEXT (Greedy) ===

First Citizen: Before we proceed any further, hear me speak.

CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the se


In [37]:
import torch
import torch.nn as nn
import numpy as np
import pickle
import editdistance

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open("char_to_idx.pkl", "rb") as f:
    char_to_idx = pickle.load(f)
with open("idx_to_char.pkl", "rb") as f:
    idx_to_char = pickle.load(f)

vocab_size = len(char_to_idx)

model = DecoderOnlyTransformer(
    vocab_size,
    d_model=256,
    nhead=8,
    num_layers=6,
    max_len=256,
    d_ff=1024,
    dropout=0.1
).to(DEVICE)

model.load_state_dict(torch.load("best_model.pth", map_location=DEVICE))
model.eval()

val_data = np.load("val.npy")
if val_data.ndim > 1:
    val_data = val_data.reshape(-1)

print(f"Validation tokens: {len(val_data):,}")

def compute_perplexity(model, data, num_batches=50, seq_len=256, batch_size=32):
    criterion = nn.CrossEntropyLoss()
    total_loss = 0.0
    with torch.no_grad():
        for _ in range(num_batches):
            idx = torch.randint(len(data) - seq_len, (batch_size,))
            x_list = [torch.from_numpy(data[i:i+seq_len]).long() for i in idx]
            y_list = [torch.from_numpy(data[i+1:i+seq_len+1]).long() for i in idx]

            x = torch.stack(x_list).to(DEVICE)
            y = torch.stack(y_list).to(DEVICE)

            logits = model(x)
            loss = criterion(logits.reshape(-1, vocab_size), y.reshape(-1))
            total_loss += loss.item()

    avg_loss = total_loss / num_batches
    ppl = torch.exp(torch.tensor(avg_loss))
    return avg_loss, ppl.item()

val_loss, val_ppl = compute_perplexity(model, val_data, num_batches=50, batch_size=32)
print(f"Validation Loss: {val_loss:.4f} | Perplexity: {val_ppl:.2f}")

def greedy_decode(model, start_text, char_to_idx, idx_to_char, max_length=800):
    model.eval()
    generated = [char_to_idx[ch] for ch in start_text if ch in char_to_idx]
    input_ids = torch.tensor([generated], dtype=torch.long, device=DEVICE)

    for _ in range(max_length):
        with torch.no_grad():
            logits = model(input_ids)
            next_token = torch.argmax(logits[0, -1]).item()

        generated.append(next_token)
        if len(generated) > 256:
            generated = generated[-256:]
        input_ids = torch.tensor([generated], dtype=torch.long, device=DEVICE)

    return "".join(idx_to_char[i] for i in generated)

prompt = "First Citizen: Before we proceed any further, hear me speak."
generated_text = greedy_decode(model, prompt, char_to_idx, idx_to_char, max_length=800)

print("\n=== GENERATED TEXT ===\n")
print(generated_text[:1500] + ("..." if len(generated_text) > 1500 else ""))

def compute_ttr(text):
    tokens = list(text)
    return len(set(tokens)) / len(tokens) if tokens else 0.0

ttr = compute_ttr(generated_text)
print(f"\nType-Token Ratio (TTR): {ttr:.4f}")

val_text = "".join(idx_to_char[int(i)] for i in val_data[:2000])
cer = editdistance.eval(generated_text[:len(val_text)], val_text) / max(len(val_text), 1)
print(f"Character Error Rate (approx on prefix): {cer:.4f}")

Validation tokens: 138,496
Validation Loss: 1.2707 | Perplexity: 3.56

=== GENERATED TEXT ===

the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the sea
CAMILLO:
I will not be so so so soul that the se

Type-Token Ratio (TTR): 0.0820
Character Error Rate (approx on prefix): 0.8930
